# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/peddikotlahimani/Flyrank-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item (one webpage), for one calendar month. I'm using fact_content_daily_performance filtered to month=2026-03 (a mid-panel month, not the sealed final month), then grouping the daily rows up to one row per content item for that month.

In [64]:
from google.colab import userdata
import duckdb

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"""
CREATE SECRET hf_token (
    TYPE huggingface,
    TOKEN '{userdata.get("HF_TOKEN")}'
);
""")
print("Connected.")

Connected.


In [65]:
columns_check = con.sql("""
    DESCRIBE SELECT * FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
""").df()
print(columns_check.to_string())

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

In [66]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

grain_check = con.sql("""
    SELECT content_hash_id, report_date, COUNT(*) c
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    GROUP BY content_hash_id, report_date
    HAVING c > 1
    LIMIT 5
""").df()
print("Rows where (content_hash_id, report_date) repeats:", grain_check.shape[0])
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows where (content_hash_id, report_date) repeats: 0


,content_hash_id,report_date,c


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features (safe to use): clicks, impressions, ctr, gsc_avg_position, engagement_rate — all directly observed in March, nothing borrowed from later.
Label / proxy: no single required label for this lane (Lane 1 doesn't force one). Where I compare groups, I use a within-month proxy: clicks_2nd_half < clicks_1st_half (built only from March data).
Context (never a feature): content_id, client_id — pseudonymous IDs, used only to group/join rows, never as model input.
Excluded: trend_direction, trend_pct — excluded because they're computed from the same performance change I might use to build a proxy label; including them risks leaking the answer into the analysis. Also excluding ga4_data_available = FALSE rows entirely — zero-filled GA4 values there aren't real zeros, just missing data.

In [67]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All

feature_plan = {
    "features": ["gsc_clicks", "gsc_impressions", "ctr (computed)", "gsc_avg_position", "engagement_rate (computed)"],
    "proxy_label": "clicks_2nd_half < clicks_1st_half (within-March only)",
    "context_only": ["content_hash_id", "client_hash_id"],
    "excluded": "no trend_pct/trend_direction columns exist in this warehouse table (those were starter-CSV-only); instead excluding rows where ga4_data_available IS FALSE, since those GA4 fields are zero-filled, not real zeros"
}
print(feature_plan)

{'features': ['gsc_clicks', 'gsc_impressions', 'ctr (computed)', 'gsc_avg_position', 'engagement_rate (computed)'], 'proxy_label': 'clicks_2nd_half < clicks_1st_half (within-March only)', 'context_only': ['content_hash_id', 'client_hash_id'], 'excluded': 'no trend_pct/trend_direction columns exist in this warehouse table (those were starter-CSV-only); instead excluding rows where ga4_data_available IS FALSE, since those GA4 fields are zero-filled, not real zeros'}


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Four checks below, each verifying a specific claim made in Sections 1–2: (1) grain holds after monthly aggregation, (2) row counts and date span match the docs, (3) missingness/availability pattern, (4) per-client row counts, to check for unbalanced history.

In [68]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

monthly = con.sql("""
    SELECT content_hash_id, client_hash_id,
           AVG(gsc_clicks) AS avg_clicks,
           SUM(gsc_clicks) AS total_clicks,
           SUM(gsc_impressions) AS total_impressions,
           AVG(gsc_avg_position) AS avg_position,
           SUM(ga4_engaged_sessions) AS engaged_sessions,
           SUM(ga4_sessions) AS sessions,
           SUM(CASE WHEN report_date >= '2026-03-16' THEN gsc_clicks ELSE 0 END) AS clicks_2nd_half,
           SUM(CASE WHEN report_date < '2026-03-16' THEN gsc_clicks ELSE 0 END) AS clicks_1st_half
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    WHERE ga4_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()

monthly['ctr'] = monthly['total_clicks'] / monthly['total_impressions']
monthly['engagement_rate'] = monthly['engaged_sessions'] / monthly['sessions']

grain_check_2 = monthly.groupby(['content_hash_id', 'client_hash_id']).size()
print("Rows where content_hash_id+client_hash_id repeats after aggregation:", (grain_check_2 > 1).sum())
print("Total monthly rows:", monthly.shape[0])
monthly.describe()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows where content_hash_id+client_hash_id repeats after aggregation: 0
Total monthly rows: 90489


,avg_clicks,total_clicks,total_impressions,avg_position,engaged_sessions,sessions,clicks_2nd_half,clicks_1st_half,ctr,engagement_rate
count,90489.000000,90489.000000,90489.000000,63856.000000,90489.000000,90489.000000,90489.000000,90489.000000,63856.000000,90237.000000
mean,0.442041,4.367426,943.061809,13.374350,0.326570,14.364265,2.786792,1.580634,0.030873,0.025977
std,1.227806,27.934660,4701.289398,13.939442,1.660332,47.007411,17.088185,12.041584,0.121153,0.110565
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,4.321690,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,24.000000,8.000000,0.000000,2.000000,0.000000,0.000000,0.003413,0.000000
75%,0.666667,2.000000,304.000000,18.350304,0.000000,8.000000,1.000000,0.000000,0.013363,0.000000
max,182.838710,5668.000000,617124.000000,305.500000,224.000000,2730.000000,3273.000000,2395.000000,1.000000,1.000000


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data can never tell me:
Unbalanced history — clients differ in how much history they have (per dim_clients.gsc_data_start/ga4_data_start), so March 2026 isn't equally representative across all clients — some may have joined recently. Check 4 (per-client row counts) shows this directly if row counts vary sharply between clients.
GSC-only early rows — some rows have search data (GSC: impressions, clicks, position) but not yet real engagement data (GA4), because a client's GSC connection can predate their GA4 connection. I already filter these out with ga4_data_available IS TRUE, but that means my sample slightly excludes clients/pages still in that early GSC-only period — my numbers describe only "GA4-active" pages, not the full universe of pages.
Window overlaps — the separate fact_content_query_90d table uses a rolling 90-day window that overlaps into other months. I'm not using that table here, but if I ever join it with this March-only data, I'd need to use only its *_prev30-style columns to avoid pulling in information from outside March.

Because of these limits, findings here are observational, not experimental — I can report that a signal moves together with an outcome, not that it causes it, and results should be read as directional/decision-support, not a guaranteed universal rule

In [69]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
gsc_only_check = con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS FALSE THEN 1 ELSE 0 END) AS gsc_only_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS full_data_rows
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
""").df()
print(gsc_only_check)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  gsc_only_rows  full_data_rows
0     9841378      1718348.0        413966.0


## Self-check
Before you submit, confirm each line honestly:

[yes] Every section above is filled — markdown thinking AND the code that backs it
[yes] The notebook runs top to bottom with no errors (Runtime → Run all)
[yes] No client names, URLs, or private queries anywhere
[yes] My claims use careful words: observed, measured, directional, decision-support
[yes] Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.